<a href="https://colab.research.google.com/github/lestermartin/starburst-dataframes-exploration/blob/main/LazyExecution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing eager & lazy execution models with dataframe operations

This module will compare & contrast the run-time characteristics of these different dataframe implementation approaches.

![Garfield](https://somospnt.com/images/blog/cover/243-fetchtype-con-jpa.jpg)

*   **Eager execution** using [Pandas](https://pandas.pydata.org/)
*   **Lazy execution** using [Ibis](https://ibis-project.org/)


# But first, some review...

## What is a dataframe?

Dataframes are two-dimensional, tabular data structures with labeled & typed columns where each column can be a different data type (numbers, strings, dates, etc), but values within a column are typically the same type.

<img src="https://pandas.pydata.org/docs/_images/01_table_dataframe.svg" alt="A mushroom-head robot drinking bubble tea" width="65%" height="65%">

**Conceptually**, think of a variable holding some, or all, of the rows from a database table.

## Example dataframe

The following table visualizes a dataframe consisting of rows describing penguins that are part of a research project.

| species | island | bill_length_mm | bill_depth_mm | flipper_length_mm | body_mass_g | sex | year |
|---|---|---|---|---|---|---|---|
| Adelie | Torgersen | 39.1 | 18.7 | 181 | 3750 | male | 2007 |
| Adelie | Torgersen | 39.5 | 17.4 | 186 | 3800 | female | 2007 |
| Adelie | Biscoe | 40.6 | 18.6 | 183 | 3550 | male | 2008 |
| Adelie | Dream | 36.6 | 17.8 | 185 | 3700 | female | 2008 |
| Gentoo | Biscoe | 46.1 | 13.2 | 211 | 4500 | female | 2009 |
| Gentoo | Biscoe | 50.0 | 16.3 | 230 | 5700 | male | 2009 |
| Chinstrap | Dream | 46.5 | 17.9 | 192 | 3500 | female | 2008 |
| Chinstrap | Dream | 50.0 | 19.5 | 196 | 3900 | male | 2008 |
| Gentoo | Biscoe | 45.2 | 14.8 | 212 | 5200 | female | 2007 |
| Adelie | Dream | 37.3 | 16.8 | 182 | 3400 | female | 2009 |


## Dataframe APIs

![api](https://static0.howtogeekimages.com/wordpress/wp-content/uploads/2018/03/api-defined-as-application-program-interface.jpeg?q=50&fit=crop&w=300&h=175&dpr=1.5$0)

Many of the functions available to dataframe objects are referred to as **transformation functions**. This means they **return a new dataframe** object or alter the dataframe is some way.

The nuance of that last statement hinges on the topic we are looking at the run-time characteristics of; **eager vs. lazy execution**.

### Eager execution

This is the typical model most learn about when they start to program. As a program executes line-by-line and it encounters a dataframe function that retrieves or modifies data in some way, it happens immediately.

*  The dataframe's contents live in memory and are accessible by the single process space
*  The dataframe is mutable and transformation functions can make needed modifications to the dataframe's contents


### Lazy execution

As you move into distributed compute engines, such as [Apache Spark](https://spark.apache.org/) and Trino, **the actual I/O required to retrieve or modify data is deferred as long as possible**.

*  Each dataframe object is actually a *set of instructions* of how to get and modify data when it actually begins the execution of the I/O activities *--they do **NOT** contain the actual data*
*  Dataframes are immutable so any modification requires the creation of a new dataframe
*  Work is actually triggered when I/O **needs** to occur; such as displaying or persisting final results
*  The processing itself is executed on, and coordinated across, multiple nodes within a cluster

**It is critical that you understand this notion of lazy execution for this module. Please revisit the prerequisite module if this refresher is not clear.**

For those that "get it", but want a deeper dive into what's going on in these distributed engines, check out the materials presented in this [video series](https://lestermartin.blog/2025/04/22/trino-query-plan-analysis-video-series/).

![query plan](https://i0.wp.com/lestermartin.blog/wp-content/uploads/2025/04/TrinoQueryPlan2.png?resize=768%2C320&ssl=1)


## Show me some code again...

Remember, we've already explored both of the frameworks and their APIs, but here is a quick comparison accessing the penguins data shown earlier. Both examples aggregate on the intersection of species & island columns.

```python
# EAGER execution                       |  # LAZY execution
import pandas as pd                     |  import ibis
                                        |  
df = pd.read_csv("penguins.csv")        |  t = ibis.read_csv("penguins.csv")
                                        |  
result = (                              |  result = (
    df.groupby(["species", "island"])   |      t.group_by(["species", "island"])
      .size()                           |           .agg(count=ibis._.count())
      .reset_index(name="count")        |       .order_by("count")
      .sort_values("count")             |  )
)                                       |  
                                        |  # nothing runs until this line
                                        |  #  returns a pandas dataframe
                                        |  df = result.execute()
```


# Env setup

The hands-on examples in this notebook access data via [Trino](https://trino.io/) and its [TPC-H connector](https://trino.io/docs/current/connector/tpch.html). If needed, here are two simple ways to set up such an environment.

*  Run [Trino in a Docker container](https://trino.io/docs/current/installation/containers.html) on your workstation
*  Register for a [Starburst Galaxy](https://www.starburst.io/starburst-galaxy/) hosted environment


## Input your Trino connection details

In [ ]:
# grab credentials from the notebook user to be used when making a connection
my_host = input("Host name")
my_username = input("User name")
my_password = getpass.getpass("Password")

## Setup and verify [Trino Python client](https://github.com/trinodb/trino-python-client)

In [ ]:
# install Trino Python client

%pip install trino

In [ ]:
# boiler-plate code for setup

from trino.dbapi import connect
from trino.auth import BasicAuthentication

conn = connect(
    host=my_host,
    port="443",
    user=my_username,
    auth=BasicAuthentication(my_username, my_password),
    http_scheme="https",
    catalog="tpch",
    schema="sf1",
)

# sanity check
print('\n Make sure the phrase ** CONNECTION IS GOOD ** displays \n')

cur = conn.cursor()
cur.execute("SELECT '** CONNECTION IS GOOD **'")
rows = cur.fetchall()
print(rows)

## Setup and verify [Ibis](https://ibis-project.org/backends/trino) for Trino backend

NOTE: Ibis can run against many different SQL engines, not just Trino.

In [ ]:
# install Ibis

%pip install 'ibis-framework[trino]'

In [ ]:
# boiler-plate code for setup

import os
import ibis
from trino.auth import BasicAuthentication

ibis.options.interactive = True

user = my_username
trino_auth_obj = BasicAuthentication(my_username, my_password)
host = my_host
port = "443"
http_scheme = "https"
catalog = "tpch"
schema = "sf1"

con = ibis.trino.connect(
    user=user, auth=trino_auth_obj, host=host, port=port, http_scheme=http_scheme, database=catalog, schema=schema
)

# sanity check
print('\n Make sure the phrase ** CONNECTION IS GOOD ** displays \n')

con.sql("select '** CONNECTION IS GOOD **' as conn_check")

# Use case

You have a schema which includes these 3 tables.

```
┌────────────────────────┐
│        NATION          │
├────────────────────────┤
│ PK  nationkey          │
│     name               │
│     ...addt cols...    │
└────────────────────────┘
            ▲
            │ 1
            │
            │ N
┌────────────────────────┐
│        CUSTOMER        │
├────────────────────────┤
│ PK  custkey            │
│     acctbal            │
│ FK  nationkey          │
│     ...addt cols...    │
└────────────────────────┘
            ▲
            │ 1
            │
            │ N
┌────────────────────────┐
│         ORDERS         │
├────────────────────────┤
│ PK  orderkey           │
│ FK  custkey            │
│     orderstatus        │
│     orderpriority      │
│     totalprice         │
│     ...addt cols...    │
└────────────────────────┘
```

For all customers with an account balance > $9900, find the total number of open orders and their average price at the unique intersection of country and order priority values; ordering them by that same intersection point.

# Build interactively with Pandas

In [ ]:
import time
import pandas as pd

# Create the hashmap (dictionary) with operation_name -> operation_time
pandas_ops = {}

start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "create a pandas DF with customer data"
# ----------------------------------------------------------
# ----------------------------------------------------------
cur.execute("SELECT * FROM customer")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_init_cust_df = pd.DataFrame(rows, columns=col_name)
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise


# display a few rows
p_init_cust_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "filter out customers with smaller balances"
# ----------------------------------------------------------
# ----------------------------------------------------------
p_lrg_bal_cust_df = p_init_cust_df[p_init_cust_df['acctbal'] > 9900.0]
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_lrg_bal_cust_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "reduce & rename customer columns"
# ----------------------------------------------------------
# ----------------------------------------------------------
p_limited_cols_cust_df = p_lrg_bal_cust_df[['custkey','nationkey']]

p_best_cust_df = p_limited_cols_cust_df.rename(columns={'custkey': 'c_custkey', 'nationkey': 'c_nationkey'})
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_best_cust_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "create a pandas DF with nation data"
# ----------------------------------------------------------
# ----------------------------------------------------------
cur.execute("SELECT * FROM nation")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_init_nation_df = pd.DataFrame(rows, columns=col_name)
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_init_nation_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "reduce & rename nation columns"
# ----------------------------------------------------------
# ----------------------------------------------------------
p_limited_cols_nation_df = p_init_nation_df.drop(columns=['regionkey', 'comment'])

p_best_nation_df = p_limited_cols_nation_df.rename(columns={'name': 'n_name', 'nationkey': 'n_nationkey'})
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_best_nation_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "join customers and nations together"
# ----------------------------------------------------------
# ----------------------------------------------------------
p_init_join_c_n_df = p_best_cust_df.merge(p_best_nation_df, left_on='c_nationkey', right_on='n_nationkey')

p_best_join_c_n_df = p_init_join_c_n_df.drop(columns=['c_nationkey', 'n_nationkey'])
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_best_join_c_n_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "create a pandas DF with orders data"
# ----------------------------------------------------------
cur.execute("SELECT * FROM orders")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_init_orders_df = pd.DataFrame(rows, columns=col_name)
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_init_orders_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "reduce & rename orders columns"
# ----------------------------------------------------------
# ----------------------------------------------------------
p_limited_cols_orders_df = p_init_orders_df[['custkey', 'orderstatus', 'orderpriority', 'totalprice']]

p_rn_lim_cols_orders_df = p_limited_cols_orders_df.rename(columns={'custkey': 'o_custkey'})
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_rn_lim_cols_orders_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "only keep open orders"
# ----------------------------------------------------------
# ----------------------------------------------------------
p_best_orders_df = p_rn_lim_cols_orders_df[p_rn_lim_cols_orders_df['orderstatus'] == 'O']
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_best_orders_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "join customers-nations with orders"
# ----------------------------------------------------------
# ----------------------------------------------------------
p_init_join_o_cn_df = p_best_orders_df.merge(p_best_join_c_n_df, left_on='o_custkey', right_on='c_custkey')

p_best_full_join_df = p_init_join_o_cn_df.drop(columns=['o_custkey'])
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_best_full_join_df.head(3)

In [ ]:
start_time = time.time() #noise

# ----------------------------------------------------------
op_name = "calculate the aggregates"
# ----------------------------------------------------------
# ----------------------------------------------------------
result = (
    p_best_full_join_df.groupby(["n_name", "orderpriority"])
    .agg(
        total_count=("c_custkey", "count"),
        avg_price=("totalprice", "mean")
    )
    .reset_index()
    .sort_values(["n_name", "orderpriority"])
)
# ----------------------------------------------------------
# ----------------------------------------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise


# display the final results
result

In [ ]:
# --- List the entire hashmap ---
print("All Pandas operations:")
for operation_name, operation_time in pandas_ops.items():
    print(f"  {round(operation_time, 4)}s -- {operation_name}")

# --- Total up the operation_time values ---
total_time = sum(pandas_ops.values())
print(f"\nTotal Pandas operation time: {round(total_time, 4)}s")

# Tidy up the Pandas code

In [ ]:
start_time = time.time()

# create a pandas DF with customer data
cur.execute("SELECT * FROM customer")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_init_cust_df = pd.DataFrame(rows, columns=col_name)

# filter out customers with smaller balances AND reduce & rename columns
p_cust_df = p_init_cust_df[p_init_cust_df['acctbal'] > 9900.0][['custkey','nationkey']] \
  .rename(columns={'custkey': 'c_custkey', 'nationkey': 'c_nationkey'})

# create a pandas DF with nation data; including cleaning up cols
cur.execute("SELECT * FROM nation")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_nation_df = pd.DataFrame(rows, columns=col_name).drop(columns=['regionkey', 'comment']) \
  .rename(columns={'name': 'n_name', 'nationkey': 'n_nationkey'})

# join customers and nations together
p_cust_nation_df = p_cust_df.merge(p_nation_df, left_on='c_nationkey', right_on='n_nationkey') \
  .drop(columns=['c_nationkey', 'n_nationkey'])

# create a pandas DF with orders data
cur.execute("SELECT * FROM orders")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_init_orders_df = pd.DataFrame(rows, columns=col_name)

# only keep open orders AND reduce & rename columns
p_orders_df = p_init_orders_df[p_init_orders_df['orderstatus'] == 'O'][['custkey', 'orderstatus', 'orderpriority', 'totalprice']] \
  .rename(columns={'custkey': 'o_custkey'})

# join customers-nations with orders AND calc the aggregates
result = p_orders_df.merge(p_cust_nation_df, left_on='o_custkey', right_on='c_custkey') \
  .drop(columns=['o_custkey']).groupby(["n_name", "orderpriority"]) \
  .agg(total_count=("c_custkey", "count"), avg_price=("totalprice", "mean")) \
  .reset_index() \
  .sort_values(["n_name", "orderpriority"])

print(f"Tidied up version took {round(time.time() - start_time, 4)} secs")

# display first 10 rows
result.head(10)

# Port to Ibis


In [ ]:
start_time = time.time()

# create an ibis DF with customer data
i_init_cust_df = con.table("customer")

# filter out customers with smaller balances AND reduce columns
i_cust_df = i_init_cust_df.filter(i_init_cust_df["acctbal"] > 9900.0) \
  .select("custkey", "acctbal", "nationkey") \

# create an ibis DF with nation data; including cleaning up cols
i_nation_df = con.table("nation") \
  .drop("regionkey", "comment") \
  .rename(
      dict(
          nation_name="name",
          n_nationkey="nationkey"
      )
  )

# join customers and nations together
i_cust_nation_df = i_cust_df.join(i_nation_df,
      i_cust_df.nationkey == i_nation_df.n_nationkey) \
  .drop("nationkey", "n_nationkey")

# create an ibis DF with orders data
i_init_orders_df = con.table("orders")

# only keep open orders AND reduce & rename columns
i_orders_df = i_init_orders_df.filter(i_init_orders_df["orderstatus"] == "O") \
  .select("custkey", "orderstatus", "orderpriority", "totalprice") \
  .rename(
      dict(
          o_custkey="custkey"
      )
  )

# join customers-nations with orders AND calc the aggregates
result = i_orders_df.join(i_cust_nation_df,
      i_orders_df.o_custkey == i_cust_nation_df.custkey) \
  .group_by(["nation_name", "orderpriority"]) \
  .aggregate(
      total_count=i_cust_nation_df.custkey.count(),
      avg_price=i_orders_df.totalprice.mean()
  ).order_by(["nation_name", "orderpriority"])

# this time is how long it took Trino to make a plan and the
#  client is starting to wait for results (watch notebook clock)
print(f"Ibis port took {round(time.time() - start_time, 4)} secs")

# display first 10 rows
result[0:10]

# Parting thoughts

**Eager Pandas wins for:**

*  Small-to-medium datasets that comfortably fit in memory
*  Exploratory, interactive work in notebooks where you want instant results
*  The enormous ecosystem of libraries that expect a pandas DataFrame as input (scikit-learn, matplotlib, statsmodels, etc.)

**Lazy Ibis wins for:**

*  Data too large for a single machine (warehouse-scale, or just bigger than RAM)
*  Writing analysis code once and running it against different backends in dev vs. prod (e.g. DuckDB locally, Snowflake/BigQuery in production) with minimal changes
*  Wanting SQL-level query optimization/pushdown instead of pulling everything into Python first
